In [1]:
import os

#os.environ['DRJIT_LIBLLVM_PATH'] = "/home/aizi0357/environments/sionna_v0/lib/libLLVM-20.so"
#os.environ["LD_LIBRARY_PATH"] = "$LD_LIBRARY_PATH:/home/aizi0357/environments/sionna_v0/lib/"

gpu_num = "" # Use "" to use the CPU
os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import sionna

resolution = [480,320] # increase for higher quality of renderings

# Allows to exit cell execution in Jupyter
class ExitCell(Exception):
    def _render_traceback_(self):
        pass

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print("gpus:", gpus)
print(f"Using GPU {gpu_num}")
if gpu_num and gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)

tf.get_logger().setLevel('ERROR')

tf.random.set_seed(1) # Set global random seed for reproducibility

#%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import time

# Import Sionna RT components
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, Camera

gpus: []
Using GPU 


In [2]:
#===================================  SETTING THE SCENE =====================================#

dtype = tf.complex128
rdtype = tf.float64


fc = 2e9  # Se cambio la frequenza il cylinder radio diventa 12.8 / 2 per lamda giusto ??????
lamda = 3e8 / fc

print("\n==== SIMULATION PARAMETERS ===")
print(f"fc: {fc}")
print(f"lamda: {lamda}")

cylinder_radius_lamda = 12.8
nu = 8

cylinder_radius = cylinder_radius_lamda * lamda
print(f"cylinder_radius: {cylinder_radius}")

scene = load_scene(f"./models/cylinder_{cylinder_radius_lamda}_lamda/cylinder_{nu}/cylinder_{nu}.xml", dtype)

scene.frequency = fc
scene.synthetic_array = True


scene.preview();


==== SIMULATION PARAMETERS ===
fc: 2000000000.0
lamda: 0.15
cylinder_radius: 1.92


2026-08-23 11:45:17 WARN main [PLYMesh] "cylinder_8.ply": attributes without postfix are not handled for now: attribute "stl" ignored.

num_edges: tf.Tensor(0, shape=(), dtype=int32)
num_wedges: 24


In [3]:
#================================ SETTING RX and TX =========================================#

pol = 'HH'   

scene.tx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="iso",
                             polarization=pol[0],
                             dtype=dtype)

scene.rx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="iso",
                             polarization=pol[1],
                             dtype=dtype)



In [4]:
# =============================== PLACING TX ================================================== #

# Calcola a desnità di onda incidente e il modulo del campo incidente.
# Metti il TX che rispetti fronhoufer ma la potenza va normalizzta con la potenza incidente. 
# Questo per vedere come varia il campo scatterato al varire della distanza di osservazione.
# Cambia la potenza del trasmettitore in modo che il campo indidente sia sempre 1 V/m.

# Per la simulzione CST allunga il cilindro 

# Valuta il campo a distanza maggiori di 20 lamda per vedere se alte dicretizzazioni sono quele ottime 

D = cylinder_radius
#tx_dist = 2*((D*2)**2)/lamda + D # TX must be in Frahofer zone
tx_dist = 10

source_point = np.array([tx_dist, 0.0, 0.0])

tx = Transmitter(name="tx", position=source_point, dtype=dtype)

scene.remove("tx")
scene.add(tx)

coef = (4 * np.pi * tx_dist)

In [5]:
#================================ IMP DOUBLE EDGE DIFRACTION =========================================#

from sionna.rt.objects_geometry import ObjectsGeometry

obj_names = list(scene.objects.keys())

obj_geom = ObjectsGeometry(scene._solver_paths)
obj_geom.is_table_gfi = True
obj_geom.is_vertex_diffraction = True
obj_geom.is_double_diffraction = True
obj_geom.init_objects(obj_names, [None])

scene._solver_paths.vertex_diffraction.set_objects_geometry(obj_geom)
scene._solver_paths.double_diffraction.set_objects_geometry(obj_geom)

# tf.math.reduce_mean(scene._solver_paths._wedges_length) / lamda

num vertices: 16


In [6]:
#=============================== DISCRETIZATION and PARAMETERS ==============================================#

E_wav = np.sqrt(2 * (1 - np.cos(2*np.pi/nu))) * (cylinder_radius / lamda)  # edge size in wavelengths
S_wav = cylinder_radius * (1 - np.cos(np.pi / nu)) / lamda  # in wavelengths
E2divR_wav = E_wav**2 * lamda / cylinder_radius

print("\n=== DISCRETIZATION PARAMETERS ===")
print(f"E = {E_wav} * lambda")
print(f"S = {S_wav} * lambda")
print(f"E^2/R = {E2divR_wav} * lambda")

RMSE_result = []


=== DISCRETIZATION PARAMETERS ===
E = 9.796695868546298 * lambda
S = 0.9743419838555297 * lambda
E^2/R = 7.498066401624382 * lambda


In [7]:

#=========================================== SIMULATION =====================================================#


circle_radius = cylinder_radius + lamda*10


print("\n\n=== ELETTROMAGNETIC PARAMETERS ===")

r = circle_radius-cylinder_radius
 
    
#print(f"circle_of_recievers_radius: {circle_radius_i}")
print(f"Distance from the cylinder: {r}")
print(f"Far field condition: {r} >> {lamda}")
print(f"Frahnoufer: {r} > {2* D**2 /lamda}")

# ============================== PLACING RX ==================================================== #
num_points = 720

theta_arr = np.linspace(0.0, 2*np.pi, num_points, endpoint=False)

def get_circle_points(theta_arr, p1, p2):
    res = np.zeros((len(theta_arr), 3))
    res[:, 0] = p1[0] * np.cos(theta_arr) + p2[0] * np.sin(theta_arr)
    res[:, 1] = p1[1] * np.cos(theta_arr) + p2[1] * np.sin(theta_arr)
    res[:, 2] = p1[2] * np.cos(theta_arr) + p2[2] * np.sin(theta_arr)

    return res

circle_p1 = np.array([1, 0, 0]) * circle_radius
circle_p2 = np.array([0, 1, 0])  * circle_radius
receive_points = get_circle_points(theta_arr, circle_p1, circle_p2)


for i in range(len(receive_points)):
    scene.remove(f"rx_{i}")
    rx = Receiver(name=f"rx_{i}", position=receive_points[i], dtype=dtype)
    scene.add(rx)

#========================================= SCATTERED FILED =================================================#


def get_field_los_full(tx_pos, rx_poses):
    # This fuction returns the field wich there would be without any obstacle

    dist = np.linalg.norm(tx_pos - rx_poses, axis=1)
    field = 1.0 / dist / (4 * 3.141592653589793) * lamda 
    return field

field_los_full = get_field_los_full(source_point, receive_points)

    
#== Calculating the SHADOW REGION
paths = scene.compute_paths(max_depth=1,
                            method="exhaustive",
                            num_samples=1e6,
                            los = True,
                            reflection = False,
                            diffraction = False,
                            vertex_diffraction = False)

a, tau = paths.cir()
path_amplitudes = a[0,:,0,0,0,:,0]
receiver_amplitudes_los = np.abs(np.sum(path_amplitudes, axis=1)) #/ lamda

los_line_idx = np.where(receiver_amplitudes_los <= 1e-6)[0][0], np.where(receiver_amplitudes_los <= 1e-6)[0][-1] #first and last index of the shadow region
field_los_full = np.where(receiver_amplitudes_los <= 0.0, field_los_full, receiver_amplitudes_los) + 1e-6

"""
if pol == 'VV':
        field_los_full = -field_los_full
        
    from scipy.constants import c as speed_of_light
    dist = np.linalg.norm(source_point - receive_points, axis=1)
    field_los_full2 = field_los_full * np.exp(-1j * 2 * np.pi * fc * dist/speed_of_light)
"""

print(f"Shadow region: [{57.3*theta_arr[los_line_idx[0]]}° ,  {57.3*theta_arr[los_line_idx[1]]}°]")



=== ELETTROMAGNETIC PARAMETERS ===
Distance from the cylinder: 1.5
Far field condition: 1.5 >> 0.15
Frahnoufer: 1.5 > 49.152
Shadow region: [136.0100179494141° ,  224.01650015197617°]


In [8]:
for i in range(len(receive_points)):
    if i<los_line_idx[0] or i>los_line_idx[1]:
        scene.remove(f"rx_{i}")

In [9]:
scene.preview()

Renderer(camera=PerspectiveCamera(aspect=1.31, children=(DirectionalLight(intensity=0.25, position=(0.0, 0.0, …

In [12]:
# Only availabe if a preview is open

scene.render_to_file(camera="preview",
                     filename="scene.png",
                     resolution=[650,500]);
